## Practica Lab12: Aplicación del Flujo de Preprocesamiento para Machine Learning

## Dataset: Adult Income

**Download latest version**

path = kagglehub.dataset_download("uciml/adult-census-income")

---

## Objetivo

Aplicar de manera autónoma el flujo de preparación de datos para un problema de clasificación utilizando un conjunto de datos real.

Al finalizar la práctica, el estudiante será capaz de:

1. Importación de librerías
2. Carga del dataset
3. Comprensión del problema
4. Exploración inicial (EDA)
5. Correlación e hipótesis
6. Variables predictoras y objetivo
7. Tratamiento de nulos
8. One-Hot Encoding
9. Train/Test Split
10. StandardScaler
11. Árbol de Decisión
12. Predicciones
13. Accuracy
14. Feature Importance
15. Conclusiones

---

# Contexto del Problema

Una institución financiera desea analizar las características de distintos individuos para determinar qué factores están asociados con ingresos superiores a $50,000 dólares anuales.

El objetivo será construir un modelo capaz de predecir si una persona pertenece a uno de los siguientes grupos:

- (<=) 50K
- (>) 50K


## Preguntas

1. ¿Cuál es la variable objetivo?
La variable objetivo es income (ingreso).

2. ¿Qué representa dicha variable?
Representa la clase de ingresos anuales de una persona, dividida en dos categorías: si gana menos o igual a 50,000 dólares (<=50K) o si gana más de 50,000 dólares (>50K).

3. ¿Qué variables consideras que podrían influir más en el ingreso de una persona?
A priori, el nivel educativo (education.num), la edad (age), las ganancias de capital (capital.gain) y las horas trabajadas por semana (hours.per.week).

4. ¿Cuántas variables predictoras existen?
Existen 14 variables predictoras originales (antes del preprocesamiento).

5. ¿Por qué fue necesario transformar variables categóricas?
Porque los algoritmos de Machine Learning (como el Árbol de Decisión) requieren que todos los datos de entrada sean numéricos para poder realizar operaciones matemáticas y encontrar los puntos de corte óptimos.

7. ¿Cuántas columnas adicionales se generaron después del One-Hot Encoding?
El dataset original tiene 8 variables categóricas. Tras aplicar la transformación, se generan aproximadamente entre 85 y 100 columnas numéricas nuevas (variables dummy), dependiendo de si se eliminó o no la primera columna de cada categoría.

8. ¿Existen valores nulos?
Sí, en el dataset original los valores nulos no aparecen como el tradicional NaN, sino que están codificados con un carácter de interrogación ? (presentes principalmente en workclass, occupation y native.country).

9. ¿Qué variables son numéricas?
age, fnlwgt, education.num, capital.gain, capital.loss y hours.per.week.

10. ¿Qué variables son categóricas?
workclass, education, marital.status, occupation, relationship, race, sex y native.country.

11. ¿Cuántos registros quedaron en entrenamiento?
Asumiendo un dataset original de 32,561 registros y un split estándar del 80%, quedaron aproximadamente 26,048 registros.

12. ¿Cuántos registros quedaron en prueba?
Con el 20% destinado a prueba, quedaron aproximadamente 6,513 registros.

13. ¿Por qué no debemos entrenar utilizando todos los datos?
Para prevenir el sobreajuste (overfitting). Si el modelo se entrena con todos los datos, simplemente los memorizará y no seremos capaces de evaluar cómo generaliza y se comporta frente a datos nuevos no vistos.

14. ¿Cuál fue el Accuracy obtenido?
Utilizando un Árbol de Decisión estándar sobre este conjunto de datos, el Accuracy obtenido suele rondar entre el 81% y el 85%.

15. ¿Consideras que el resultado es adecuado?
Sí, es un buen punto de partida (modelo base). Sin embargo, podría mejorarse mediante la optimización de hiperparámetros o utilizando algoritmos más robustos como Random Forest.

16. ¿Qué factores podrían afectar el desempeño del modelo?
El desbalance de la clase objetivo (hay muchas más personas que ganan <= 50K), valores atípicos severos en la variable capital.gain, una mala estrategia de imputación de los nulos y permitir que el árbol crezca sin un límite de profundidad.

18. ¿Cuál fue la variable más importante?
El modelo suele destacar el estado civil (específicamente la categoría marital.status_Married-civ-spouse), seguido de capital.gain y education.num, como los factores más determinantes.

19. ¿Cuál fue la menos importante?
Las variables categóricas relacionadas con los países de origen específicos en native.country (a excepción de Estados Unidos) y otras características demográficas minoritarias terminan aportando una importancia cercana a 0.

20. ¿Coinciden los resultados con tus hipótesis iniciales?
Sí, la educación y la edad demostraron ser críticas. No obstante, suele resultar revelador el enorme peso que tiene el estado civil (estar casado) por encima del propio nivel educativo.

21. ¿Qué variables aportan más información al modelo?
Aquellas que logran reducir más la impureza de los datos al hacer las particiones (mayor ganancia de información): capital.gain, el estado civil, los años de educación y la edad.

# Entregables

El repositorio deberá contener:

```text
Notebooks/
└── Laboratorio12.ipynb
```

---


## Paso 1. Importación de librerias 

In [1]:
import pandas as pd
import numpy as np
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

## Paso 2. Carga del dataset
Se descarga la última versión desde kagglehub y se asume que el archivo descargado se llama 'adult.csv' en la ruta correspondiente

In [2]:
path = kagglehub.dataset_download("uciml/adult-census-income")

100%|██████████| 450k/450k [00:00<00:00, 544kB/s]

Extracting files...


In [3]:
df = pd.read_csv(f"{path}/adult.csv")

## Paso 3 y 4. Comprensión del problema y Exploración inicial (EDA)
Se reemplazan los valores '?' (que son nulos en este dataset) por NaN

In [4]:
df.replace('?', np.nan, inplace=True)

## Paso 5 y 6. Variables predictoras y objetivo


In [5]:
X = df.drop('income', axis=1) # Variables predictoras
y = df['income'].apply(lambda x: 1 if x == '>50K' else 0) # Variable objetivo binaria

## Paso 7. Tratamiento de nulos
Imputamos variables categóricas con la moda

In [6]:
for col in ['workclass', 'occupation', 'native.country']:
    X[col].fillna(X[col].mode()[0], inplace=True)

/tmp/ipykernel_22852/3804905809.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X[col].fillna(X[col].mode()[0], inplace=True)


## 8. One-Hot Encoding
Transformación de variables categóricas a numéricas

In [7]:
X_encoded = pd.get_dummies(X, drop_first=True)

## Paso 9. Train/Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

## Paso 10. StandardScaler

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Paso 11. Árbol de Decisión

In [10]:
clf = DecisionTreeClassifier(random_state=42, max_depth=10) # max_depth limitado para evitar sobreajuste
clf.fit(X_train_scaled, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


## Paso 12. Predicciones

In [11]:
y_pred = clf.predict(X_test_scaled)

## Paso 13. Accuracy

In [12]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy del modelo: {accuracy:.4f}")

Accuracy del modelo: 0.8537


## Paso 14. Feature Importance

In [13]:
importances = clf.feature_importances_
feature_names = X_encoded.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
print(feature_importance_df.head(5))

                              Feature  Importance
29  marital.status_Married-civ-spouse    0.382211
2                       education.num    0.203737
3                        capital.gain    0.182187
4                        capital.loss    0.067354
0                                 age    0.051399
